# Day 2 v2: LLM Preprocessing (Vietnamese) — 5 truong

**Thay doi so voi v1:** LLM rewrite **toan bo 5 truong** (Tieu de, Danh muc, Thuong hieu, Mo ta, Thong so).

- v1: LLM chi tao 2 truong (Mo ta + Thong so), title/category/brand giu nguyen tu data goc
- v2: LLM rewrite tat ca → loai SKU codes tu title, chuan hoa toan bo fields

**Input:** `SeanSunny/items_raw_tv_v5` (full = title + features + brand + category)

**Output:** `SeanSunny/items_tv_v5`

## 1. Imports + Load data

In [1]:
from litellm import completion
from dotenv import load_dotenv
import json
import os
import re

load_dotenv(override=True)

from pricer_vi.items import Item
from pricer_vi.preprocessor import Preprocessor, SYSTEM_PROMPT, build_summary
from pricer_vi.batch import Batch

from groq import Groq
groq_client = Groq(api_key=os.environ.get('GROQ_API_KEY'))

In [2]:
dataset = "SeanSunny/items_raw_tv_v6"

train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"Total: {len(items):,}")

Train: 110,000 | Val: 5,000 | Test: 5,000
Total: 120,000


In [3]:
# Assign IDs (required for batch custom_id mapping)
for index, item in enumerate(items):
    item.id = index

print(f"Assigned IDs: 0 to {len(items)-1}")

Assigned IDs: 0 to 119999


In [4]:
# Inspect raw data — full column now includes brand + category
print(f"Title: {items[0].title}")
print(f"Category: {items[0].category}")
print(f"Brand: {items[0].brand}")
print(f"Price: {items[0].price:,} VND")
print(f"\nFull text ({len(items[0].full)} chars):")
print(items[0].full[:500])

Title: Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870
Category: Điện Tử - Công Nghệ
Brand: TEEMO PC
Price: 2,376,000 VND

Full text (3104 chars):
Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC
Kính chào Quý khách, chào mừng quý khách đến với gian hàng của chúng tôi, chúc Quý khách một ngày tốt lành và mua sắm vui vẻ. Dưới đây là một số thông tin tham khảo về sản phẩm. ĐẶC ĐIỂM: TƯƠNG THÍCH... | Kính chào Quý khách, chào mừng quý khách đến với gian hàng của chúng tôi, chúc Quý khách một ngày tốt lành và mua sắm vui vẻ. Dưới đây là một số thông tin tham khảo về sản phẩm. ĐẶC ĐIỂM: TƯƠNG THÍCH VỚI TẤT CẢ MÃ


In [5]:
print(items[0].full)

Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC
Kính chào Quý khách, chào mừng quý khách đến với gian hàng của chúng tôi, chúc Quý khách một ngày tốt lành và mua sắm vui vẻ. Dưới đây là một số thông tin tham khảo về sản phẩm. ĐẶC ĐIỂM: TƯƠNG THÍCH... | Kính chào Quý khách, chào mừng quý khách đến với gian hàng của chúng tôi, chúc Quý khách một ngày tốt lành và mua sắm vui vẻ. Dưới đây là một số thông tin tham khảo về sản phẩm. ĐẶC ĐIỂM: TƯƠNG THÍCH VỚI TẤT CẢ MÃ MÁY CÓ TRONG TÊN SẢN PHẨM THÔNG SỐ KỸ THUẬT Dùng cho tất cả các mã máy có trong tên sản phẩm Công suất: Tiêu chuẩn pin theo máy chênh lệch +/- 5% Điện Áp: Tiêu chuẩn Số Cell: Tiêu chuẩn Loại Pin: Li-on. Thời gian sử dụng cho một lần sạc đầy: 2h – 4h – 6h tùy số Cell và đời máy Hàng mới full box 100%, hoàn toàn tương thích với máy Cần tư vấn thêm quý khách vui lòng nhắn tin với Shop nhé CHẾ ĐỘ BẢO HÀNH VÀ HẬU MÃI Thời gian bảo hành: 6 tháng – 12 tháng tùy model được ghi trong phần thông tin chi 

In [6]:
# Verify full column has brand + category appended
assert "Thương hiệu:" in items[0].full, "full column missing brand"
print("Full column enriched: OK")

# Check an item with SKU code
sku_pattern = re.compile(r"\b(?=[A-Z0-9]{8,}\b)(?=.*[A-Z])(?=.*\d)[A-Z0-9]+\b")
for item in items[:5000]:
    if sku_pattern.search(item.title):
        print(f"\nItem with SKU code:")
        print(f"  Title: {item.title[:100]}")
        print(f"  SKU codes: {sku_pattern.findall(item.title)}")
        break

Full column enriched: OK

Item with SKU code:
  Title: Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870
  SKU codes: ['TEBAT870']


## 2. Test single item

SYSTEM_PROMPT yeu cau LLM rewrite **5 truong**: Tieu de, Danh muc, Thuong hieu, Mo ta, Thong so.

Tat ca truong deu do LLM viet lai — giong pipeline tieng Anh.

In [7]:
print("SYSTEM_PROMPT:")
print(SYSTEM_PROMPT)

SYSTEM_PROMPT:
Tạo mô tả ngắn gọn cho một sản phẩm. Chỉ trả lời đúng 5 dòng theo định dạng sau. Không bao gồm mã sản phẩm hay mã nội bộ.
Tiêu đề: Tiêu đề ngắn gọn, chính xác
Danh mục: Phân loại sản phẩm
Thương hiệu: Tên thương hiệu
Mô tả: 1 câu mô tả sản phẩm
Thông số: 1 câu về tính năng nổi bật


In [8]:
# Test LLM on 1 item
preprocessor = Preprocessor()
summary = preprocessor.preprocess(items[0])

print(f"--- Input ---")
print(f"Title: {items[0].title}")
print(f"Category: {items[0].category}")
print(f"Brand: {items[0].brand}")
print(f"\n--- LLM Output (Summary) ---")
print(summary)
print(f"\n--- Cost ---")
print(f"Tokens: {preprocessor.total_input_tokens} in / {preprocessor.total_output_tokens} out")
print(f"Cost: ${preprocessor.total_cost:.4f}")

--- Input ---
Title: Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO PC TEBAT870
Category: Điện Tử - Công Nghệ
Brand: TEEMO PC

--- LLM Output (Summary) ---
Tiêu đề: Pin Li‑ion Dell Vostro 14 5459 – New Seal TEEMO  
Danh mục: Phụ kiện laptop  
Thương hiệu: TEEMO PC  
Mô tả: Pin thay thế hoàn toàn tương thích với Dell Vostro 14 5459, thiết kế chuẩn công suất và điện áp.  
Thông số: Công suất ±5 %, 2‑6 h sử dụng, bảo hành 6‑12 tháng tùy model.

--- Cost ---
Tokens: 1165 in / 122 out
Cost: $0.0001


In [9]:
# Test on a few more items (mix categories + SKU codes)
test_indices = [0, 100, 1000, 5000, 50000, 100000]
for idx in test_indices:
    summary = preprocessor.preprocess(items[idx])
    print(f"\n[{idx}] {items[idx].title[:60]}... | {items[idx].category}")
    print(summary)
    print("---")

print(f"\nTotal cost so far: ${preprocessor.total_cost:.4f}")


[0] Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập K... | Điện Tử - Công Nghệ
Tiêu đề: Pin Tương Thích Laptop Dell Vostro 14 5459  
Danh mục: Phụ kiện Laptop – Pin  
Thương hiệu: TEEMO PC  
Mô tả: Pin Li‑ion đa cell, công suất chuẩn, thời gian sạc 2‑6 giờ, hoàn toàn tương thích với Dell Vostro 14 5459.  
Thông số: Công suất ±5%, điện áp tiêu chuẩn, thời gian bảo hành 6‑12 tháng, khuyến nghị sạc 8‑10 giờ cho pin mới.
---

[100] Patch PVC Velcro Zombie dán ba lô túi xách... | Mẹ và Bé
Tiêu đề: Patch PVC Velcro Zombie cho Balo & Túi  
Danh mục: Phụ kiện Trang trí  
Thương hiệu: OEM  
Mô tả: Patch zombie 3D dán lót, giữ cố định, không bị biến dạng.  
Thông số: Chất liệu PVC dẻo, mặt sau có gai giúp dán nhanh lên quần áo, balo, nón.
---

[1000] Ugreen UG11672DV101TK 1M màu Đen Cáp tín hiệu DVI 24 + 1 - H... | Điện Tử - Công Nghệ
Tiêu đề: Cáp DVI 24+1 1M Ugreen Đen  
Danh mục: Cáp tín hiệu và truyền dữ liệu  
Thương hiệu: UGREEN  
Mô tả: Cáp DVI 24+1 male to male, 1m, phủ vàng, 

## 3. Full Batch Processing (120K items)

Dung `Batch` class tu `pricer_vi/batch.py`.

- 120 batches x 1000 items = 120K
- Model: `openai/gpt-oss-20b` (Groq Batch API)
- Uoc tinh chi phi: ~$5-10
- Uoc tinh thoi gian: ~6h

In [10]:
# Reset summaries from test above
for item in items:
    item.summary = None

Batch.create(items)

Created 120 batches


In [11]:
Batch.run()

  0%|          | 0/120 [00:00<?, ?it/s]

Submitted 120 batches


In [12]:
# QUAN TRONG: Save state ngay sau run() — neu kernel crash se mat batch_ids
Batch.save()

Saved 120 batches


In [13]:
# Chay cell nay NHIEU LAN cho den khi tat ca batches hoan thanh
# "Finished 120 of 120 batches" la xong
Batch.fetch()

  0%|          | 0/120 [00:00<?, ?it/s]

Finished 1 of 120 batches


In [14]:
import time                                                                                                                                            
                                                        
while True:
    Batch.fetch()
    finished = sum(1 for b in Batch.batches if b.done)                                                                                                 
    if finished == len(Batch.batches):
        print("All batches done!")                                                                                                                     
        break                                             
    print(f"Waiting 30s... ({finished}/{len(Batch.batches)})")
    time.sleep(30)  

  0%|          | 0/120 [00:00<?, ?it/s]

Finished 1 of 120 batches
Waiting 30s... (1/120)


  0%|          | 0/120 [00:00<?, ?it/s]

Finished 2 of 120 batches
Waiting 30s... (2/120)


  0%|          | 0/120 [00:00<?, ?it/s]

...

Finished 119 of 120 batches
Waiting 30s... (119/120)


  0%|          | 0/120 [00:00<?, ?it/s]

Finished 119 of 120 batches
Waiting 30s... (119/120)


  0%|          | 0/120 [00:00<?, ?it/s]

Finished 120 of 120 batches
All batches done!


In [15]:
# Save state sau khi fetch xong
Batch.save()

Saved 120 batches


### Resume (neu kernel crash giua chung)

Uncomment va chay 2 cell ben duoi neu can resume.

In [ ]:
# # Resume: load state tu file pickle
# Batch.load(items)
# Batch.fetch()

### Resubmit failed batches (neu co)

Groq co the fail 1 so batches do spend limit. Chay cell ben duoi de kiem tra va resubmit.

In [ ]:
# Check status cua tat ca batches
from collections import Counter

statuses = []
for batch in Batch.batches:
    if batch.done:
        statuses.append("done")
    else:
        result = groq_client.batches.retrieve(batch.batch_id)
        statuses.append(result.status)

print(Counter(statuses))

In [ ]:
# Resubmit failed batches
import time

resubmitted = 0
for batch in Batch.batches:
    if not batch.done:
        result = groq_client.batches.retrieve(batch.batch_id)
        if result.status in ("failed", "expired", "cancelled"):
            batch.send_file()
            batch.submit_batch()
            resubmitted += 1
            time.sleep(0.5)

print(f"Resubmitted {resubmitted} batches")
if resubmitted > 0:
    Batch.save()

## 4. Kiem tra ket qua

In [16]:
# Check missing summaries — muc tieu: 0
missing = [i for i, item in enumerate(items) if not item.summary]
print(f"Missing summaries: {len(missing)}")
if missing:
    print(f"First 10 missing IDs: {missing[:10]}")

Missing summaries: 0


In [17]:
# Xem vi du summary tu nhieu categories
sample_indices = [0, 100, 1000, 5000, 50000, 100000, 115000, 119000]
for idx in sample_indices:
    if idx < len(items) and items[idx].summary:
        print(f"\n[{idx}] Category goc: {items[idx].category} | Price: {items[idx].price:,} VND")
        print(f"Original title: {items[idx].title[:80]}")
        print(f"Summary:\n{items[idx].summary}")
        print("---")


[0] Category goc: Điện Tử - Công Nghệ | Price: 2,376,000 VND
Original title: Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO P
Summary:
Tiêu đề: Pin Tương Thích Dell Vostro 14 5459  
Danh mục: Pin Laptop  
Thương hiệu: TEEMO PC  
Mô tả: Pin Li‑ion mới, hoàn toàn tương thích với Dell Vostro 14 5459, cung cấp công suất và thời gian sạc 2‑6h tùy số Cell.  
Thông số: Được thiết kế với tiêu chuẩn công suất +/-5%, điện áp chuẩn, và bảo hành 6‑12 tháng.
---

[100] Category goc: Mẹ và Bé | Price: 59,000 VND
Original title: Patch PVC Velcro Zombie dán ba lô túi xách
Summary:
Tiêu đề: Patch PVC Velcro Zombie  
Danh mục: Phụ kiện thời trang  
Thương hiệu: OEM  
Mô tả: Patch PVC dán dễ dàng, có gai để gắn vào quần áo, túi xách và balo.  
Thông số: Chất liệu dẻo, sắc nét, không bị biến dạng khi sử dụng.
---

[1000] Category goc: Điện Tử - Công Nghệ | Price: 222,000 VND
Original title: Ugreen UG11672DV101TK 1M màu Đen Cáp tín hiệu DVI 24 + 1 - HÀNG CHÍNH HÃNG
Summary:

In [ ]:
# Thong ke: categories do LLM output
from collections import Counter

llm_categories = []
for item in items:
    if item.summary:
        for line in item.summary.split("\n"):
            line_stripped = line.strip()
            if line_stripped.startswith("Danh m\u1ee5c:"):
                cat = line_stripped.split(":", 1)[1].strip()
                llm_categories.append(cat)
                break

cat_counts = Counter(llm_categories)
print(f"Total items with category: {len(llm_categories):,}")
print(f"Unique categories: {len(cat_counts)}")
print(f"\nTop 20 categories:")
for cat, count in cat_counts.most_common(20):
    print(f"  {count:>6,} | {cat}")

## 5. Build prompts + Clean up + Push to HF Hub

In [18]:
# Build prompt from summary
for item in items:
    if item.summary:
        item.make_prompt(item.summary)

# Verify
print("Sample prompt:")
print(items[0].prompt)

Sample prompt:
Sản phẩm này giá bao nhiêu?

Tiêu đề: Pin Tương Thích Dell Vostro 14 5459  
Danh mục: Pin Laptop  
Thương hiệu: TEEMO PC  
Mô tả: Pin Li‑ion mới, hoàn toàn tương thích với Dell Vostro 14 5459, cung cấp công suất và thời gian sạc 2‑6h tùy số Cell.  
Thông số: Được thiết kế với tiêu chuẩn công suất +/-5%, điện áp chuẩn, và bảo hành 6‑12 tháng.

Giá: 2376000


In [19]:
# Clean up: remove fields not needed in final dataset
for item in items:
    item.full = None
    item.brand = None
    item.id = None

print("Cleaned up: full, brand, id = None")

Cleaned up: full, brand, id = None


In [20]:
# Push to HuggingFace Hub
username = "SeanSunny"
output_dataset = f"{username}/items_tv_v6"

# Split back into train/val/test (same order as loaded)
train_out = items[:110_000]
val_out = items[110_000:115_000]
test_out = items[115_000:120_000]

print(f"Pushing {output_dataset}...")
print(f"Train: {len(train_out):,} | Val: {len(val_out):,} | Test: {len(test_out):,}")

Item.push_to_hub(output_dataset, train_out, val_out, test_out)
print("Done!")

Pushing SeanSunny/items_tv_v6...
Train: 110,000 | Val: 5,000 | Test: 5,000


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/110 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done!


## Done!

Dataset `SeanSunny/items_tv_v5` da co summary (5 truong, toan bo do LLM rewrite).

Summary format:
```
Tieu de: [LLM rewritten — sach, khong SKU codes]
Danh muc: [LLM rewritten]
Thuong hieu: [LLM rewritten]
Mo ta: [LLM generated]
Thong so: [LLM generated]
```

Buoc tiep: Day 3 Baseline ML.